# Experiment Template

Clone this notebook for new experiments. Replace `MODEL_NAME` with your model.

In [ ]:
MODEL_NAME = 'xgboost'  # Change this

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.data.loader import load_raw_data, filter_stores
from src.data.features import add_all_features
from src.data.preprocessor import preprocess
from src.models import get_model_class
from src.utils.visualization import plot_predictions, plot_residuals

config = load_config(MODEL_NAME)
set_seed(config.get('seed', 42))
print(f'Config: {config}')

In [ ]:
# Load & preprocess data
df, _ = load_raw_data(config)
df = filter_stores(df, config)
df = add_all_features(df)
train_df, val_df, test_df, scaler = preprocess(df, config)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
# Train
model_class = get_model_class(MODEL_NAME)
model = model_class(config)
info = model.train(train_df, val_df)
print(f'Training info: {info}')

In [ ]:
# Evaluate
metrics = model.evaluate(val_df)
print(f'Validation metrics: {metrics}')

preds = model.predict(val_df)
plot_predictions(val_df['Sales'].values, preds, title=f'{MODEL_NAME} - Validation')
plot_residuals(val_df['Sales'].values, preds, title=f'{MODEL_NAME} - Residuals')

In [ ]:
# Save results
results = model.get_result_template(metrics)
out_dir = model.save_results(results, config.get('results_dir', 'results'))
print(f'Saved to {out_dir}')